# 35a — Deep Learning Tools: PyTorch

Corresponds to: Note 35 — Deep Learning Tools: PyTorch, TensorFlow, Keras

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09. TensorFlow is not installed in this environment; the TF/Keras patterns are shown in Note 35 as code blocks.

**What this notebook demonstrates:**
1. The four-line training loop — forward, loss, zero_grad, backward, step — with explicit gradient inspection
2. `nn.Module` subclassing pattern — `__init__` and `forward`, parameter registration
3. `Dataset` and `DataLoader` — batching, shuffling, iterating
4. GPU device handling — `torch.device`, `.to(device)`, CPU/GPU tensor movement
5. Model saving and loading — `state_dict`, `torch.save`, `torch.load`

## Part 1 — The Training Loop

We trace through one complete training iteration, inspecting gradients before and after `backward()` to make the automatic differentiation concrete.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

model     = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1))
optimiser = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.MSELoss()

x = torch.tensor([[1.0, 2.0]])
y = torch.tensor([[3.0]])

# ── Before backward: gradients are None ──────────────────────────────────────
print('Gradients before backward:')
for name, param in model.named_parameters():
    print(f'  {name}: {param.grad}')

# ── Forward pass ──────────────────────────────────────────────────────────────
logits = model(x)
loss   = criterion(logits, y)
print(f'\nlogits: {logits.item():.4f}')
print(f'loss:   {loss.item():.4f}')

# ── Backward pass ─────────────────────────────────────────────────────────────
optimiser.zero_grad()
loss.backward()

print('\nGradients after backward:')
for name, param in model.named_parameters():
    print(f'  {name}: {param.grad.numpy().round(4)}')

# ── Weight update ─────────────────────────────────────────────────────────────
w_before = model[0].weight.data.clone()
optimiser.step()
w_after  = model[0].weight.data

print('\nFirst layer weight change (after one SGD step):')
print((w_after - w_before).numpy().round(6))

Gradients before backward:
  0.weight: None
  0.bias: None
  2.weight: None
  2.bias: None

logits: -0.4436
loss:   11.8586

Gradients after backward:
  0.weight: [[-1.0522 -2.1044]
 [ 0.      0.    ]
 [ 0.7816  1.5632]]
  0.bias: [-1.0522  0.      0.7816]
  2.weight: [[-5.092  0.    -0.304]]
  2.bias: [-6.8873]

First layer weight change (after one SGD step):
[[ 0.105219  0.210439]
 [ 0.        0.      ]
 [-0.078161 -0.156323]]


## Part 2 — nn.Module Subclassing

The `nn.Module` pattern is the standard way to define models in PyTorch. We show how parameters are registered automatically and how `train()`/`eval()` mode affects dropout.

In [2]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.fc1     = nn.Linear(in_dim, hidden_dim)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

mlp = MLP(in_dim=4, hidden_dim=8, out_dim=2)

# Parameter count
total = sum(p.numel() for p in mlp.parameters())
print(f'Total parameters: {total}')
for name, p in mlp.named_parameters():
    print(f'  {name}: shape {tuple(p.shape)}')

# train vs eval mode
x_test = torch.ones(1, 4)
mlp.train()
out_train = mlp(x_test)
mlp.eval()
with torch.no_grad():
    out_eval = mlp(x_test)

print(f'\nOutput (train mode, dropout active): {out_train.detach().numpy().round(4)}')
print(f'Output (eval mode, dropout off):     {out_eval.numpy().round(4)}')

Total parameters: 58
  fc1.weight: shape (8, 4)
  fc1.bias: shape (8,)
  fc2.weight: shape (2, 8)
  fc2.bias: shape (2,)

Output (train mode, dropout active): [[-0.2166 -0.0412]]
Output (eval mode, dropout off):     [[-0.0498 -0.1106]]


## Part 3 — Dataset and DataLoader

We implement a custom `Dataset` and wrap it in a `DataLoader`, then iterate over batches.

In [3]:
from torch.utils.data import Dataset, DataLoader

class SyntheticDataset(Dataset):
    def __init__(self, n_samples, n_features):
        torch.manual_seed(42)
        self.X = torch.randn(n_samples, n_features)
        self.y = (self.X[:, 0] > 0).long()   # binary label from first feature

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SyntheticDataset(n_samples=100, n_features=4)
loader  = DataLoader(dataset, batch_size=16, shuffle=True)

print(f'Dataset size:   {len(dataset)}')
print(f'Number of batches: {len(loader)}')

# Inspect first batch
x_batch, y_batch = next(iter(loader))
print(f'Batch x shape:  {tuple(x_batch.shape)}')
print(f'Batch y shape:  {tuple(y_batch.shape)}')
print(f'Batch y values: {y_batch.numpy()}')

Dataset size:   100
Number of batches: 7
Batch x shape:  (16, 4)
Batch y shape:  (16,)
Batch y values: [1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 1]


## Part 4 — GPU Device Handling

We show the device detection pattern and how to move models and tensors between CPU and GPU.

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Move model and data to device
mlp_device = MLP(in_dim=4, hidden_dim=8, out_dim=2).to(device)
x_device   = torch.randn(8, 4).to(device)

print(f'Model device: {next(mlp_device.parameters()).device}')
print(f'Input device: {x_device.device}')

with torch.no_grad():
    out = mlp_device(x_device)
print(f'Output device: {out.device}')
print(f'Output shape:  {tuple(out.shape)}')

# Move result back to CPU for numpy conversion
out_cpu = out.cpu().numpy()
print(f'Output (numpy): {out_cpu.round(4)}')

Device: cpu
Model device: cpu
Input device: cpu
Output device: cpu
Output shape:  (8, 2)
Output (numpy): [[-0.4723  0.6326]
 [ 0.0359  0.0655]
 [ 0.0899 -0.3838]
 [-0.2227  0.18  ]
 [-0.0618  0.1091]
 [ 0.0549  0.0479]
 [-0.261  -0.1201]
 [ 0.1924 -0.2374]]


## Part 5 — Saving and Loading Models

We save the model's `state_dict`, reload it into a fresh model, and verify the outputs match.

In [5]:
import tempfile, os

# Save
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'model.pth')
    torch.save(mlp.state_dict(), path)
    print(f'Saved to: {path}')

    # Load into a fresh model
    mlp_loaded = MLP(in_dim=4, hidden_dim=8, out_dim=2)
    mlp_loaded.load_state_dict(torch.load(path, weights_only=True))
    mlp_loaded.eval()

    # Verify outputs match
    x_check = torch.randn(3, 4)
    mlp.eval()
    with torch.no_grad():
        out_original = mlp(x_check)
        out_loaded   = mlp_loaded(x_check)

    match = torch.allclose(out_original, out_loaded)
    print(f'Outputs match after reload: {match}')

Saved to: C:\Users\SAJADK~1\AppData\Local\Temp\tmpjg7_2xeb\model.pth


Outputs match after reload: True


## Summary

This notebook demonstrated:
1. The four-line training loop — forward → loss → zero_grad → backward → step — with gradient inspection before and after `backward()`
2. `nn.Module` subclassing — automatic parameter registration, `train()`/`eval()` mode switching for dropout
3. `Dataset` and `DataLoader` — custom dataset, batching, shuffling, iterating
4. GPU device handling — `torch.device`, `.to(device)`, moving results back to CPU for NumPy
5. Model saving and loading — `state_dict`, `torch.save`, `torch.load`, output verification